# Manual Annotation Tool for Ground Truth Rep Counting

This notebook provides an interactive interface for manually annotating exercise repetitions in IMU data to create accurate ground truth labels.

## Features:
- **Interactive visualization** of all 12 IMU channels
- **Manual rep boundary marking** with click-and-drag
- **Auto-detection assistance** using peak detection
- **Quality validation** and consistency checks
- **Optimal windowing determination** based on actual rep durations
- **Export annotated dataset** for training

In [1]:
# Import required libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from matplotlib.widgets import Button, Slider, SpanSelector
import json
from datetime import datetime
from scipy.signal import butter, filtfilt, find_peaks
import pickle
from IPython.display import display, clear_output
import ipywidgets as widgets

plt.style.use('default')
%matplotlib widget

print("Manual Annotation Tool Loaded")
print("Ready for precise ground truth annotation!")

Manual Annotation Tool Loaded
Ready for precise ground truth annotation!


## 1. Initialize Annotation System

In [5]:
class GroundTruthAnnotator:
    def __init__(self, data_folder="./Data/BMI270/Ex1", meta_path="SportMeta.xlsx"):
        self.data_folder = data_folder
        self.meta_path = meta_path
        self.current_file_idx = 0
        self.annotations = {}
        self.rep_boundaries = []
        self.current_data = None
        self.current_metadata = None
        
        # Load metadata
        self.load_metadata()
        
        # Load existing annotations
        self.annotation_file = "ground_truth_annotations.json"
        self.load_existing_annotations()
        
        print(f"Initialized annotator with {len(self.meta_df)} files")
        print(f"Existing annotations: {len(self.annotations)} files")
    
    def load_metadata(self):
        """Load exercise metadata"""
        try:
            self.meta_df = pd.read_excel(self.meta_path)
            print(f"Loaded metadata: {len(self.meta_df)} entries")
            print(f"Activities: {self.meta_df['activity'].value_counts().to_dict()}")
        except Exception as e:
            print(f"Error loading metadata: {e}")
            self.meta_df = pd.DataFrame()
    
    def load_existing_annotations(self):
        """Load existing annotations if available"""
        if os.path.exists(self.annotation_file):
            try:
                with open(self.annotation_file, 'r') as f:
                    self.annotations = json.load(f)
                print(f"Loaded {len(self.annotations)} existing annotations")
            except Exception as e:
                print(f"Error loading annotations: {e}")
                self.annotations = {}
    
    def save_annotations(self):
        """Save annotations to JSON file"""
        try:
            with open(self.annotation_file, 'w') as f:
                json.dump(self.annotations, f, indent=2)
            print(f"✅ Saved annotations for {len(self.annotations)} files")
        except Exception as e:
            print(f"❌ Error saving annotations: {e}")
    
    def parse_bmi270_data(self, filepath):
        """Parse BMI270 IMU data from CSV file"""
        def convert_uint_to_int(n):
            return n - 0x2000 if n >= 0x2000 else n
        
        def get_bmi270_from_str(srt):
            if len(srt) != 36:
                return None
            try:
                channels = []
                for i in range(12):
                    start_idx = i * 3
                    hex_val = int(srt[start_idx:start_idx+3], 16)
                    channels.append(convert_uint_to_int(hex_val))
                return channels
            except:
                return None
        
        try:
            with open(filepath, 'r') as f:
                lines = f.readlines()
            
            data = []
            for line in lines:
                line = line.strip()
                if line and len(line) == 36:
                    parsed = get_bmi270_from_str(line)
                    if parsed:
                        data.append(parsed)
            
            return np.array(data) if data else None
        except Exception as e:
            print(f"Error parsing {filepath}: {e}")
            return None
    
    def apply_lowpass_filter(self, data, cutoff=20, fs=100, order=4):
        """Apply Butterworth low-pass filter"""
        nyquist = 0.5 * fs
        normal_cutoff = cutoff / nyquist
        b, a = butter(order, normal_cutoff, btype='low', analog=False)
        
        filtered_data = np.zeros_like(data)
        for i in range(data.shape[1]):
            filtered_data[:, i] = filtfilt(b, a, data[:, i])
        
        return filtered_data

# Initialize the annotator
annotator = GroundTruthAnnotator()
print("\n🎯 Ground Truth Annotator Ready!")

Loaded metadata: 96 entries
Error loading metadata: 'activity'
Initialized annotator with 0 files
Existing annotations: 0 files

🎯 Ground Truth Annotator Ready!


## 2. File Selection and Loading

In [6]:
# Create file selection widget
file_options = [(f"File {row['file_id']:05d}: {row['activity']} by {row['pid']}", idx) 
               for idx, (_, row) in enumerate(annotator.meta_df.iterrows())]

file_selector = widgets.Dropdown(
    options=file_options,
    description='Select File:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='400px')
)

load_button = widgets.Button(
    description='Load File',
    button_style='primary',
    layout=widgets.Layout(width='100px')
)

file_info = widgets.HTML(value="Select a file to start annotation")

def load_selected_file(button):
    global current_file_data, current_metadata, rep_boundaries
    
    file_idx = file_selector.value
    row = annotator.meta_df.iloc[file_idx]
    file_id = row['file_id']
    activity = row['activity']
    participant = row['pid']
    
    filepath = os.path.join(annotator.data_folder, f"DI_{file_id:05d}.CSV")
    
    if not os.path.exists(filepath):
        file_info.value = f"❌ File not found: {filepath}"
        return
    
    # Load and process data
    raw_data = annotator.parse_bmi270_data(filepath)
    if raw_data is None:
        file_info.value = f"❌ Failed to parse: {filepath}"
        return
    
    # Apply filtering
    filtered_data = annotator.apply_lowpass_filter(raw_data, cutoff=20, fs=100)
    
    # Store current data
    current_file_data = filtered_data
    current_metadata = {
        'file_id': file_id,
        'activity': activity,
        'participant': participant,
        'filepath': filepath,
        'duration_seconds': len(filtered_data) / 100.0,
        'total_samples': len(filtered_data)
    }
    
    # Load existing annotations if available
    file_key = str(file_id)
    if file_key in annotator.annotations:
        rep_boundaries = annotator.annotations[file_key]['rep_boundaries'].copy()
        existing_reps = annotator.annotations[file_key]['num_reps']
        file_info.value = f"✅ Loaded: {activity} by {participant} | {len(filtered_data)} samples | {len(filtered_data)/100:.1f}s | Existing: {existing_reps} reps"
    else:
        rep_boundaries = []
        file_info.value = f"✅ Loaded: {activity} by {participant} | {len(filtered_data)} samples | {len(filtered_data)/100:.1f}s | No existing annotation"
    
    print(f"Loaded File {file_id}: {activity} by {participant}")
    print(f"Duration: {current_metadata['duration_seconds']:.1f}s, Samples: {len(filtered_data)}")

load_button.on_click(load_selected_file)

display(widgets.HBox([file_selector, load_button]))
display(file_info)

# Initialize global variables
current_file_data = None
current_metadata = None
rep_boundaries = []

HTML(value='Select a file to start annotation')

TypeError: Cannot index by location index with a non-integer key

## 3. Interactive Visualization and Annotation

In [ ]:
def create_annotation_plot():
    """Create interactive plot for manual annotation"""
    if current_file_data is None:
        print("❌ Please load a file first")
        return
    
    # Create figure with subplots for all channels
    fig, axes = plt.subplots(4, 3, figsize=(18, 12))
    fig.suptitle(f"File {current_metadata['file_id']}: {current_metadata['activity']} "
                f"by {current_metadata['participant']} | Duration: {current_metadata['duration_seconds']:.1f}s", 
                fontsize=14)
    
    # Time axis in seconds
    time_axis = np.arange(len(current_file_data)) / 100.0
    
    # Channel names
    channel_names = [
        'AccXL', 'AccYL', 'AccZL', 'GyroXL', 'GyroYL', 'GyroZL',
        'AccXR', 'AccYR', 'AccZR', 'GyroXR', 'GyroYR', 'GyroZR'
    ]
    
    # Plot each channel
    for i in range(12):
        row = i // 3
        col = i % 3
        ax = axes[row, col]
        
        # Plot signal
        ax.plot(time_axis, current_file_data[:, i], 'b-', linewidth=1, alpha=0.8)
        ax.set_title(f'{channel_names[i]}', fontsize=11, fontweight='bold')
        ax.set_xlabel('Time (s)', fontsize=9)
        ax.set_ylabel('Amplitude', fontsize=9)
        ax.grid(True, alpha=0.3)
        
        # Plot existing rep boundaries
        for boundary in rep_boundaries:
            ax.axvline(x=boundary/100.0, color='red', linestyle='--', linewidth=2, alpha=0.7)
        
        # Highlight primary channel (AccYL)
        if i == 1:  # AccYL
            ax.set_facecolor('#f0f8ff')
            ax.set_title(f'{channel_names[i]} (PRIMARY)', fontsize=11, fontweight='bold', color='darkblue')
    
    plt.tight_layout()
    return fig, axes

# Button to create/refresh plot
plot_button = widgets.Button(
    description='Create Annotation Plot',
    button_style='success',
    layout=widgets.Layout(width='200px')
)

def show_plot(button):
    if current_file_data is not None:
        fig, axes = create_annotation_plot()
        plt.show()
    else:
        print("❌ Please load a file first")

plot_button.on_click(show_plot)
display(plot_button)

## 4. Manual Boundary Addition Tools

In [ ]:
# Manual boundary input
boundary_input = widgets.FloatText(
    value=0.0,
    description='Time (s):',
    step=0.1,
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='150px')
)

add_boundary_btn = widgets.Button(
    description='Add Boundary',
    button_style='warning',
    layout=widgets.Layout(width='120px')
)

clear_boundaries_btn = widgets.Button(
    description='Clear All',
    button_style='danger',
    layout=widgets.Layout(width='100px')
)

auto_detect_btn = widgets.Button(
    description='Auto-Detect',
    button_style='info',
    layout=widgets.Layout(width='120px')
)

boundary_status = widgets.HTML(value="No boundaries set")

def add_boundary(button):
    global rep_boundaries
    if current_file_data is None:
        boundary_status.value = "❌ No file loaded"
        return
    
    time_sec = boundary_input.value
    sample_idx = int(time_sec * 100)
    
    if 0 <= sample_idx < len(current_file_data):
        rep_boundaries.append(sample_idx)
        rep_boundaries = sorted(list(set(rep_boundaries)))  # Remove duplicates and sort
        
        num_reps = len(rep_boundaries) - 1 if len(rep_boundaries) > 1 else 0
        boundary_status.value = f"✅ Added boundary at {time_sec:.1f}s | Total boundaries: {len(rep_boundaries)} | Reps: {num_reps}"
    else:
        boundary_status.value = f"❌ Invalid time: {time_sec:.1f}s (range: 0-{len(current_file_data)/100:.1f}s)"

def clear_boundaries(button):
    global rep_boundaries
    rep_boundaries = []
    boundary_status.value = "🗑️ Cleared all boundaries"

def auto_detect_boundaries(button):
    global rep_boundaries
    if current_file_data is None:
        boundary_status.value = "❌ No file loaded"
        return
    
    # Use AccYL (channel 1) for peak detection
    signal = current_file_data[:, 1]
    
    # Find peaks with adaptive threshold
    prominence_threshold = np.std(signal) * 1.5
    peaks, properties = find_peaks(signal, prominence=prominence_threshold, distance=75)
    
    # Convert peaks to boundaries
    rep_boundaries = []
    
    if len(peaks) > 0:
        # Add start boundary
        rep_boundaries.append(0)
        
        # Add boundaries between peaks
        for i in range(len(peaks) - 1):
            midpoint = (peaks[i] + peaks[i+1]) // 2
            rep_boundaries.append(midpoint)
        
        # Add end boundary
        rep_boundaries.append(len(signal) - 1)
    
    rep_boundaries = sorted(list(set(rep_boundaries)))
    num_reps = len(rep_boundaries) - 1 if len(rep_boundaries) > 1 else 0
    boundary_status.value = f"🤖 Auto-detected {len(peaks)} peaks → {len(rep_boundaries)} boundaries → {num_reps} reps"

add_boundary_btn.on_click(add_boundary)
clear_boundaries_btn.on_click(clear_boundaries)
auto_detect_btn.on_click(auto_detect_boundaries)

display(widgets.HBox([boundary_input, add_boundary_btn, clear_boundaries_btn, auto_detect_btn]))
display(boundary_status)

## 5. Annotation Review and Quality Check

In [ ]:
def review_current_annotation():
    """Review current annotation with detailed statistics"""
    if current_file_data is None:
        print("❌ No file loaded")
        return
    
    print("\n" + "="*60)
    print("📋 ANNOTATION REVIEW")
    print("="*60)
    
    print(f"📄 File: {current_metadata['file_id']} - {current_metadata['activity']} by {current_metadata['participant']}")
    print(f"⏱️  Duration: {current_metadata['duration_seconds']:.1f}s ({current_metadata['total_samples']} samples)")
    print(f"🔢 Boundaries: {len(rep_boundaries)}")
    
    if len(rep_boundaries) < 2:
        print("⚠️  Need at least 2 boundaries to define repetitions")
        return
    
    # Calculate rep durations
    rep_durations = []
    for i in range(len(rep_boundaries) - 1):
        duration = (rep_boundaries[i+1] - rep_boundaries[i]) / 100.0
        rep_durations.append(duration)
    
    num_reps = len(rep_durations)
    avg_duration = np.mean(rep_durations)
    std_duration = np.std(rep_durations)
    
    print(f"🏃 Repetitions: {num_reps}")
    print(f"📊 Rep Duration: {avg_duration:.2f}s ± {std_duration:.2f}s")
    print(f"📈 Range: {min(rep_durations):.2f}s - {max(rep_durations):.2f}s")
    
    # Quality checks
    print("\n🔍 Quality Checks:")
    
    # Check for very short or long reps
    short_reps = [d for d in rep_durations if d < 0.5]
    long_reps = [d for d in rep_durations if d > 10.0]
    
    if short_reps:
        print(f"⚠️  {len(short_reps)} very short reps (<0.5s): {[f'{d:.2f}s' for d in short_reps]}")
    if long_reps:
        print(f"⚠️  {len(long_reps)} very long reps (>10s): {[f'{d:.2f}s' for d in long_reps]}")
    
    # Check for high variability
    cv = std_duration / avg_duration if avg_duration > 0 else 0
    if cv > 0.5:
        print(f"⚠️  High variability in rep durations (CV={cv:.2f})")
    
    if not short_reps and not long_reps and cv <= 0.5:
        print("✅ Quality checks passed")
    
    # Windowing recommendations
    optimal_window_size = min(max(64, int(avg_duration * 100 * 1.5)), 256)
    optimal_stride = optimal_window_size // 4
    
    print(f"\n🎯 Recommended Windowing:")
    print(f"   Window Size: {optimal_window_size} samples ({optimal_window_size/100:.2f}s)")
    print(f"   Stride: {optimal_stride} samples ({optimal_stride/100:.2f}s)")
    print(f"   Overlap: {((optimal_window_size - optimal_stride) / optimal_window_size * 100):.1f}%")
    print(f"   Update Rate: {100/optimal_stride:.1f} Hz")
    
    # Detailed rep breakdown
    print(f"\n📝 Detailed Rep Breakdown:")
    for i, duration in enumerate(rep_durations):
        start_time = rep_boundaries[i] / 100.0
        end_time = rep_boundaries[i+1] / 100.0
        print(f"   Rep {i+1}: {start_time:.2f}s - {end_time:.2f}s ({duration:.2f}s)")

review_btn = widgets.Button(
    description='Review Annotation',
    button_style='info',
    layout=widgets.Layout(width='150px')
)

review_btn.on_click(lambda b: review_current_annotation())
display(review_btn)

## 6. Save and Export Functions

In [ ]:
def save_current_annotation():
    """Save current annotation to database"""
    if current_metadata is None:
        print("❌ No file loaded")
        return
    
    if len(rep_boundaries) < 2:
        print("❌ Need at least 2 boundaries to save annotation")
        return
    
    file_key = str(current_metadata['file_id'])
    
    # Calculate rep statistics
    rep_durations = []
    for i in range(len(rep_boundaries) - 1):
        duration = (rep_boundaries[i+1] - rep_boundaries[i]) / 100.0
        rep_durations.append(duration)
    
    num_reps = len(rep_durations)
    avg_duration = np.mean(rep_durations)
    
    # Determine optimal windowing
    optimal_window_size = min(max(64, int(avg_duration * 100 * 1.5)), 256)
    optimal_stride = optimal_window_size // 4
    
    # Create annotation record
    annotation = {
        'file_id': current_metadata['file_id'],
        'activity': current_metadata['activity'],
        'participant': current_metadata['participant'],
        'rep_boundaries': rep_boundaries,
        'num_reps': num_reps,
        'rep_durations': rep_durations,
        'avg_rep_duration': avg_duration,
        'std_rep_duration': np.std(rep_durations),
        'optimal_window_size': optimal_window_size,
        'optimal_stride': optimal_stride,
        'annotation_timestamp': datetime.now().isoformat(),
        'total_duration': current_metadata['duration_seconds'],
        'total_samples': current_metadata['total_samples']
    }
    
    # Save to annotations database
    annotator.annotations[file_key] = annotation
    annotator.save_annotations()
    
    print(f"✅ Saved annotation for File {current_metadata['file_id']}:")
    print(f"   • Reps: {num_reps}")
    print(f"   • Avg Duration: {avg_duration:.2f}s")
    print(f"   • Optimal Window: {optimal_window_size} samples")
    print(f"   • Optimal Stride: {optimal_stride} samples")

def export_training_dataset():
    """Export all annotations as training dataset"""
    if not annotator.annotations:
        print("❌ No annotations to export")
        return
    
    print("\n🚀 Exporting Training Dataset...")
    print("="*50)
    
    training_data = []
    activity_stats = {}
    
    for file_key, annotation in annotator.annotations.items():
        file_id = annotation['file_id']
        activity = annotation['activity']
        num_reps = annotation['num_reps']
        window_size = annotation['optimal_window_size']
        stride = annotation['optimal_stride']
        
        # Load file data
        filepath = os.path.join(annotator.data_folder, f"DI_{file_id:05d}.CSV")
        raw_data = annotator.parse_bmi270_data(filepath)
        if raw_data is None:
            print(f"⚠️  Skipping {file_id}: Cannot load data")
            continue
        
        filtered_data = annotator.apply_lowpass_filter(raw_data, cutoff=20, fs=100)
        
        # Create windows with optimal parameters
        windows = []
        for i in range(0, len(filtered_data) - window_size + 1, stride):
            window = filtered_data[i:i + window_size]
            windows.append(window)
        
        # Add to training data
        for window in windows:
            training_data.append({
                'window': window,
                'exercise_label': activity,
                'rep_count': num_reps,
                'file_id': file_id,
                'window_size': window_size,
                'stride': stride,
                'participant': annotation['participant']
            })
        
        # Update statistics
        if activity not in activity_stats:
            activity_stats[activity] = {
                'files': 0,
                'total_reps': 0,
                'total_windows': 0,
                'avg_rep_duration': [],
                'window_sizes': [],
                'strides': []
            }
        
        stats = activity_stats[activity]
        stats['files'] += 1
        stats['total_reps'] += num_reps
        stats['total_windows'] += len(windows)
        stats['avg_rep_duration'].append(annotation['avg_rep_duration'])
        stats['window_sizes'].append(window_size)
        stats['strides'].append(stride)
        
        print(f"✅ Processed File {file_id}: {len(windows)} windows")
    
    # Save training dataset
    dataset_filename = f'ground_truth_training_data_{datetime.now().strftime("%Y%m%d_%H%M%S")}.pkl'
    with open(dataset_filename, 'wb') as f:
        pickle.dump(training_data, f)
    
    # Create comprehensive report
    report = {
        'dataset_info': {
            'total_files': len(annotator.annotations),
            'total_windows': len(training_data),
            'creation_timestamp': datetime.now().isoformat(),
            'dataset_filename': dataset_filename
        },
        'activity_statistics': {},
        'windowing_summary': {}
    }
    
    # Calculate activity statistics
    for activity, stats in activity_stats.items():
        report['activity_statistics'][activity] = {
            'files': stats['files'],
            'total_reps': stats['total_reps'],
            'total_windows': stats['total_windows'],
            'avg_rep_duration': np.mean(stats['avg_rep_duration']),
            'std_rep_duration': np.std(stats['avg_rep_duration']),
            'avg_window_size': np.mean(stats['window_sizes']),
            'avg_stride': np.mean(stats['strides']),
            'windows_per_file': stats['total_windows'] / stats['files']
        }
    
    # Save report
    report_filename = f'ground_truth_report_{datetime.now().strftime("%Y%m%d_%H%M%S")}.json'
    with open(report_filename, 'w') as f:
        json.dump(report, f, indent=2)
    
    # Print summary
    print(f"\n🎉 Export Complete!")
    print(f"📊 Dataset: {len(training_data)} windows from {len(annotator.annotations)} files")
    print(f"💾 Saved: {dataset_filename}")
    print(f"📋 Report: {report_filename}")
    
    print(f"\n📈 Activity Breakdown:")
    for activity, stats in report['activity_statistics'].items():
        print(f"   {activity}: {stats['files']} files, {stats['total_reps']} reps, {stats['total_windows']} windows")
        print(f"      Avg rep duration: {stats['avg_rep_duration']:.2f}s")
        print(f"      Avg window: {stats['avg_window_size']:.0f} samples, stride: {stats['avg_stride']:.0f}")

# Control buttons
save_btn = widgets.Button(
    description='Save Annotation',
    button_style='success',
    layout=widgets.Layout(width='140px')
)

export_btn = widgets.Button(
    description='Export Dataset',
    button_style='primary',
    layout=widgets.Layout(width='140px')
)

save_btn.on_click(lambda b: save_current_annotation())
export_btn.on_click(lambda b: export_training_dataset())

display(widgets.HBox([save_btn, export_btn]))

## 7. Annotation Progress Overview

In [ ]:
def show_annotation_progress():
    """Show overall annotation progress"""
    total_files = len(annotator.meta_df)
    annotated_files = len(annotator.annotations)
    progress_pct = (annotated_files / total_files * 100) if total_files > 0 else 0
    
    print("\n" + "="*60)
    print("📊 ANNOTATION PROGRESS OVERVIEW")
    print("="*60)
    
    print(f"📁 Total Files: {total_files}")
    print(f"✅ Annotated: {annotated_files} ({progress_pct:.1f}%)")
    print(f"⏳ Remaining: {total_files - annotated_files}")
    
    if annotator.annotations:
        # Activity breakdown
        activity_counts = {}
        total_reps = 0
        
        for annotation in annotator.annotations.values():
            activity = annotation['activity']
            if activity not in activity_counts:
                activity_counts[activity] = {'files': 0, 'reps': 0}
            activity_counts[activity]['files'] += 1
            activity_counts[activity]['reps'] += annotation['num_reps']
            total_reps += annotation['num_reps']
        
        print(f"\n🏃 Total Repetitions Annotated: {total_reps}")
        print(f"\n📋 By Activity:")
        for activity, counts in activity_counts.items():
            print(f"   {activity}: {counts['files']} files, {counts['reps']} reps")
        
        # Show next files to annotate
        annotated_file_ids = set(int(k) for k in annotator.annotations.keys())
        remaining_files = []
        
        for idx, row in annotator.meta_df.iterrows():
            if row['file_id'] not in annotated_file_ids:
                remaining_files.append((idx, row['file_id'], row['activity'], row['pid']))
        
        if remaining_files:
            print(f"\n⏭️  Next Files to Annotate:")
            for i, (idx, file_id, activity, pid) in enumerate(remaining_files[:5]):
                print(f"   {i+1}. File {file_id:05d}: {activity} by {pid}")
            if len(remaining_files) > 5:
                print(f"   ... and {len(remaining_files) - 5} more")
    else:
        print("\n⚠️  No annotations created yet")

progress_btn = widgets.Button(
    description='Show Progress',
    button_style='info',
    layout=widgets.Layout(width='140px')
)

progress_btn.on_click(lambda b: show_annotation_progress())
display(progress_btn)

# Show initial progress
show_annotation_progress()

## 8. Usage Instructions

### 🎯 **Manual Annotation Workflow:**

1. **Select File**: Use the dropdown to choose a file to annotate
2. **Load File**: Click "Load File" to process the IMU data
3. **Create Plot**: Click "Create Annotation Plot" to visualize all channels
4. **Mark Boundaries**: 
   - Use "Auto-Detect" for initial boundary detection
   - Manually add boundaries by entering time (in seconds) and clicking "Add Boundary"
   - Focus on the **AccYL (PRIMARY)** channel for clearest rep patterns
5. **Review**: Click "Review Annotation" to check quality and get windowing recommendations
6. **Save**: Click "Save Annotation" to store the annotation
7. **Repeat**: Continue with next files
8. **Export**: Click "Export Dataset" when done to create training data

### 📊 **Quality Guidelines:**
- **Boundaries should mark start/end of each repetition**
- **Rep durations should be consistent** (CV < 0.5)
- **Avoid very short (<0.5s) or very long (>10s) reps**
- **Use AccYL channel as primary reference** for vertical movements

### 🎉 **Output:**
- **ground_truth_annotations.json**: All annotations
- **ground_truth_training_data_YYYYMMDD_HHMMSS.pkl**: Training dataset
- **ground_truth_report_YYYYMMDD_HHMMSS.json**: Comprehensive statistics

The exported dataset will have **precise rep counts** and **optimal window/stride parameters** determined from your manual annotations!